# set 4 salinity-location figures before trial 4.5

this notebook loads the saved three-dimensional salinity outputs for trials 4.0, 4.1, 4.2, and 4.4. it generates the same location plot for each available simulation and saves a comparison figure.

trial 4.3 was a diagnostic rather than a new simulation, so it has no salinity output. the notebook generates a separate schematic of the original and corrected mixing boxes for that trial.

the saved trial 4.4 file contains only day 0 and day 1. its plot is therefore labeled day 1 rather than day 12.

In [ ]:
using Oceananigans
using Oceananigans.Units
using Oceananigans.OutputReaders: FieldTimeSeries, OnDisk
using CairoMakie

In [ ]:
# find the set 4 directory whether julia starts in the repository or this folder
function find_set4_directory(starting_directory)
    directory = abspath(starting_directory)

    for step in 1:8
        if isdir(joinpath(directory, "4.0")) && isdir(joinpath(directory, "4.1"))
            return directory
        end

        candidate = joinpath(directory, "debugging", "4")
        if isdir(joinpath(candidate, "4.0"))
            return candidate
        end

        parent_directory = dirname(directory)
        if parent_directory == directory
            break
        end
        directory = parent_directory
    end

    error("could not find the debugging/4 directory")
end

set4_directory = find_set4_directory(pwd())
figure_directory = joinpath(set4_directory, "figures_pre4_5")
mkpath(figure_directory)

trial_files = [
    (id = "4.0", filename = "amazon_rivers_off_dye_test_trial4_0_ordinary_redi_rivers_off_0p1deg_salinity_3d.jld2"),
    (id = "4.1", filename = "amazon_rivers_on_dye_test_trial4_1_river_3x3_0p1deg_salinity_3d.jld2"),
    (id = "4.2", filename = "amazon_rivers_on_dye_test_trial4_2_river_3x3_kz0p1_30m_0p1deg_salinity_3d.jld2"),
    (id = "4.4", filename = "amazon_rivers_on_dye_test_trial4_4_river_3x3_kz0p1_measured_mouth_0p1deg_salinity_3d.jld2")
]

@show set4_directory
@show figure_directory

In [ ]:
# find the lowest active-ocean salinity in every horizontal water column
function minimum_in_each_column(values)
    Nx, Ny, Nz = size(values)
    column_minimum = fill(NaN, Nx, Ny)

    for j in 1:Ny
        for i in 1:Nx
            found_water = false
            minimum_value = Inf

            for k in 1:Nz
                value = values[i, j, k]

                # inactive cells are stored as zero in these output files
                if value != 0
                    found_water = true
                    minimum_value = min(minimum_value, value)
                end
            end

            if found_water
                column_minimum[i, j] = minimum_value
            end
        end
    end

    return column_minimum
end

# load the last saved salinity record and calculate its location statistics
function load_trial(trial)
    path = joinpath(set4_directory, trial.id, trial.filename)
    series = FieldTimeSeries(path, "S"; backend = OnDisk())
    record = length(series.times)
    field = series[record]
    values = Array(interior(field))
    longitude, latitude, depth = nodes(field)

    column_minimum = minimum_in_each_column(values)

    overall_minimum = Inf
    minimum_index = (1, 1, 1)
    negative_cell_count = 0

    for index in CartesianIndices(values)
        value = values[index]
        if value != 0
            if value < overall_minimum
                overall_minimum = value
                minimum_index = Tuple(index)
            end
            if value < 0
                negative_cell_count += 1
            end
        end
    end

    negative_column_indices = findall(value -> isfinite(value) && value < 0, column_minimum)
    low_column_count = count(value -> isfinite(value) && value < 25, column_minimum)

    i, j, k = minimum_index

    return (;
        id = trial.id,
        day = series.times[record] / days,
        record_count = record,
        longitude = collect(longitude),
        latitude = collect(latitude),
        depth = collect(depth),
        column_minimum,
        overall_minimum,
        minimum_longitude = longitude[i],
        minimum_latitude = latitude[j],
        minimum_depth = depth[k],
        negative_column_indices,
        negative_cell_count,
        low_column_count
    )
end

In [ ]:
trial_data = [load_trial(trial) for trial in trial_files]

for trial in trial_data
    @info "saved salinity result" trial=trial.id day=trial.day minimum_salinity=trial.overall_minimum negative_cells=trial.negative_cell_count columns_below_25=trial.low_column_count records=trial.record_count
end

In [ ]:
# plot the full domain and a closer view of the amazon mouth
function salinity_location_figure(trial)
    day_label = string(round(trial.day, digits = 2))
    figure = Figure(size = (1250, 590))

    Label(
        figure[0, 1:3],
        "trial $(trial.id): lowest salinity in each water column on day $(day_label)",
        fontsize = 22
    )

    full_axis = Axis(
        figure[1, 1],
        xlabel = "longitude (degrees east)",
        ylabel = "latitude (degrees north)",
        title = "full regional domain"
    )

    zoom_axis = Axis(
        figure[1, 2],
        xlabel = "longitude (degrees east)",
        ylabel = "latitude (degrees north)",
        title = "amazon mouth zoom"
    )

    heatmap = heatmap!(
        full_axis,
        trial.longitude,
        trial.latitude,
        trial.column_minimum;
        colorrange = (-3, 32),
        colormap = :haline
    )

    heatmap!(
        zoom_axis,
        trial.longitude,
        trial.latitude,
        trial.column_minimum;
        colorrange = (-3, 32),
        colormap = :haline
    )

    finite_values = filter(isfinite, vec(trial.column_minimum))
    if minimum(finite_values) < 25 && maximum(finite_values) > 25
        contour!(full_axis, trial.longitude, trial.latitude, trial.column_minimum; levels = [25.0], color = :white, linewidth = 2)
        contour!(zoom_axis, trial.longitude, trial.latitude, trial.column_minimum; levels = [25.0], color = :white, linewidth = 2)
    end

    if !isempty(trial.negative_column_indices)
        negative_longitudes = [trial.longitude[index[1]] for index in trial.negative_column_indices]
        negative_latitudes = [trial.latitude[index[2]] for index in trial.negative_column_indices]

        scatter!(full_axis, negative_longitudes, negative_latitudes; marker = :rect, markersize = 15, color = :red)
        scatter!(zoom_axis, negative_longitudes, negative_latitudes; marker = :rect, markersize = 15, color = :red)
    end

    scatter!(full_axis, [trial.minimum_longitude], [trial.minimum_latitude]; marker = :star5, markersize = 18, color = :red, strokecolor = :white, strokewidth = 1)
    scatter!(zoom_axis, [trial.minimum_longitude], [trial.minimum_latitude]; marker = :star5, markersize = 18, color = :red, strokecolor = :white, strokewidth = 1)

    xlims!(zoom_axis, -52, -47)
    ylims!(zoom_axis, -2, 2.5)

    Colorbar(figure[1, 3], heatmap, label = "minimum salinity in column")

    footer = "minimum = $(round(trial.overall_minimum, digits = 5)) at ($(round(trial.minimum_longitude, digits = 2)), $(round(trial.minimum_latitude, digits = 2))), depth $(round(abs(trial.minimum_depth), digits = 2)) m | $(trial.negative_cell_count) negative 3-d cells | $(trial.low_column_count) columns below 25"
    Label(figure[2, 1:3], footer, fontsize = 14)

    return figure
end

In [ ]:
# generate and save every available pre-4.5 salinity-location figure
trial_figures = Dict{String, Figure}()

for trial in trial_data
    figure = salinity_location_figure(trial)
    trial_figures[trial.id] = figure

    day_tag = replace(string(round(trial.day, digits = 2)), "." => "p")
    trial_tag = replace(trial.id, "." => "_")
    output_path = joinpath(figure_directory, "trial_$(trial_tag)_day_$(day_tag)_salinity_locations.png")

    save(output_path, figure, px_per_unit = 1.5)
    @info "saved figure" output_path
end

trial_figures["4.1"]

In [ ]:
# put the four available simulation results on one shared color scale
function make_comparison_figure(trials)
    figure = Figure(size = (1300, 950))
    Label(figure[0, 1:3], "pre-4.5 salinity-location comparison", fontsize = 24)
    last_heatmap = nothing

    for (number, trial) in enumerate(trials)
        row = number <= 2 ? 1 : 2
        column = number <= 2 ? number : number - 2
        title = "trial $(trial.id), day $(round(trial.day, digits = 2)): min $(round(trial.overall_minimum, digits = 3))"
        axis = Axis(figure[row, column], xlabel = "longitude", ylabel = "latitude", title = title)

        last_heatmap = heatmap!(axis, trial.longitude, trial.latitude, trial.column_minimum; colorrange = (-3, 32), colormap = :haline)
        xlims!(axis, -52, -47)
        ylims!(axis, -2, 2.5)

        finite_values = filter(isfinite, vec(trial.column_minimum))
        if minimum(finite_values) < 25 && maximum(finite_values) > 25
            contour!(axis, trial.longitude, trial.latitude, trial.column_minimum; levels = [25.0], color = :white, linewidth = 2)
        end

        if !isempty(trial.negative_column_indices)
            negative_longitudes = [trial.longitude[index[1]] for index in trial.negative_column_indices]
            negative_latitudes = [trial.latitude[index[2]] for index in trial.negative_column_indices]
            scatter!(axis, negative_longitudes, negative_latitudes; marker = :rect, markersize = 14, color = :red)
        end

        scatter!(axis, [trial.minimum_longitude], [trial.minimum_latitude]; marker = :star5, markersize = 16, color = :red, strokecolor = :white, strokewidth = 1)
    end

    Colorbar(figure[1:2, 3], last_heatmap, label = "minimum salinity in column")
    Label(figure[3, 1:3], "white outline: below 25 | red square: negative column | red star: overall minimum | trial 4.4 is day 1 because later records are unavailable", fontsize = 14)

    return figure
end

comparison_figure = make_comparison_figure(trial_data)
comparison_path = joinpath(figure_directory, "trials_4_0_to_4_4_salinity_locations.png")
save(comparison_path, comparison_figure, px_per_unit = 1.5)
comparison_figure

In [ ]:
# trial 4.3 did not run the model; this schematic shows the diagnostic change
function box_outline(center_longitude, center_latitude, half_width)
    longitude = [
        center_longitude - half_width,
        center_longitude + half_width,
        center_longitude + half_width,
        center_longitude - half_width,
        center_longitude - half_width
    ]

    latitude = [
        center_latitude - half_width,
        center_latitude - half_width,
        center_latitude + half_width,
        center_latitude + half_width,
        center_latitude - half_width
    ]

    return longitude, latitude
end

original_longitude, original_latitude = box_outline(-49.5, 0.16, 0.15)
candidate_longitude, candidate_latitude = box_outline(-49.35, -0.15, 0.25)

diagnostic_figure = Figure(size = (850, 650))
axis = Axis(
    diagnostic_figure[1, 1],
    xlabel = "longitude (degrees east)",
    ylabel = "latitude (degrees north)",
    title = "trial 4.3 mixing-box diagnostic schematic",
    aspect = DataAspect()
)

lines!(axis, original_longitude, original_latitude; linewidth = 4, label = "trial 4.2 box: 13.58% coverage")
lines!(axis, candidate_longitude, candidate_latitude; linewidth = 4, label = "candidate box: 88.30% coverage")
scatter!(axis, [-49.35], [-0.15]; marker = :star5, markersize = 22, color = :red, label = "strongest forced cell")
axislegend(axis, position = :rt)
xlims!(axis, -49.75, -49.0)
ylims!(axis, -0.5, 0.45)

Label(diagnostic_figure[2, 1], "schematic only: trial 4.3 analyzed the forcing and did not create a new salinity field", fontsize = 14)

diagnostic_path = joinpath(figure_directory, "trial_4_3_mixing_box_diagnostic.png")
save(diagnostic_path, diagnostic_figure, px_per_unit = 1.5)
diagnostic_figure